# 🩺 Dermatology Disease Classification — Machine Learning Project

## 🎯 Project Objective
Build a **multi-class machine learning classification system** that predicts the dermatology disease class from the available clinical and histopathological features.

### 📌 Project Workflow
This notebook follows the existing workflow from **data loading → preprocessing → EDA → train/test split → SMOTE → scaling → model training → model comparison → evaluation → feature importance → model saving → prediction**.

> **Important:** The existing project steps are preserved in the same order. The additions at the end are optional project extensions for improving analysis, validation, interpretability, and deployment readiness.

### 🧰 Technologies Used
- Python
- Pandas & NumPy
- Matplotlib & Seaborn
- Scikit-learn
- Imbalanced-learn (SMOTE)
- Pickle for model persistence

### 🏥 Dataset Type
The project uses a dermatology dataset containing clinical/histopathological attributes and a target column named **`class`**.

### ⚠️ Scope
This is an **academic machine-learning project**. Model predictions should not be treated as a medical diagnosis or as a replacement for evaluation by a qualified healthcare professional.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## 1. 📂 Load the Dataset

Load the dermatology dataset into a Pandas DataFrame and perform the first inspection.

**Purpose:** Verify that the dataset is available and ready for preprocessing.


In [ ]:
data = pd.read_csv("dermotology_data.csv")
data.head()

In [ ]:
data.columns

In [ ]:
data.shape

In [ ]:
data.info()

### 📝 Dataset Note

- The dataset contains **34 input features** and one target column named **`class`**.
- The **`age`** column may contain the value `?`.
- The `age` column is converted to numeric form.
- Missing values are handled before model training.

This preprocessing step prevents non-numeric values from causing errors during model development.


In [ ]:
data["age"] = pd.to_numeric(data["age"], errors="coerce")

print("Missing values:")
print(data.isnull().sum())

In [ ]:
data.describe()

## 2. 🔎 Check Duplicate Records

Identify duplicate rows and remove them before further analysis.

**Why this matters:** Duplicate observations can introduce bias and make evaluation results less representative.


In [ ]:
print("Duplicate rows:", data.duplicated().sum())

data = data.drop_duplicates()
print("Shape after removing duplicates:", data.shape)

## 3. 🎯 Target Class Distribution

Inspect how many observations belong to each dermatology disease class.

**Why this matters:** Unequal class frequencies can affect model learning and motivate the later use of **SMOTE on the training data**.


In [ ]:
print(data["class"].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x="class", data=data)
plt.title("Distribution of Dermatology Disease Classes")
plt.xlabel("Disease Class")
plt.ylabel("Number of Samples")
plt.show()

## 4. 🧹 Handle Missing Values

Handle missing values after converting the `age` column to numeric format.

**Strategy used in this notebook:** Missing `age` values are replaced with the **median age**.

> The test set is kept separate from training transformations later in the workflow.


In [ ]:
# Fill missing age values with the median age
data["age"] = data["age"].fillna(data["age"].median())

print("Total missing values after treatment:", data.isnull().sum().sum())

## 5. 📊 Exploratory Data Analysis (EDA)

Explore feature distributions and relationships before model training.

### EDA Goals
- Understand feature distributions.
- Identify unusual patterns.
- Examine relationships between numerical features.
- Inspect feature correlations.

EDA helps explain the dataset and provides useful context for the classification task.


In [ ]:
# Plot distributions of numerical features
numeric_columns = data.drop(columns=["class"]).columns

fig, axes = plt.subplots(5, 7, figsize=(20, 18))
axes = axes.flatten()

for i, column in enumerate(numeric_columns):
    sns.histplot(data[column], ax=axes[i], kde=True)
    axes[i].set_title(column.replace("_", " "), fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18,12))
sns.heatmap(data.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

## 6. 🧩 Separate Input Features and Target

Split the dataset into:

- **`X`** → input features
- **`y`** → target disease class

This creates the feature matrix and target vector required by scikit-learn classifiers.


In [ ]:
X = data.drop("class", axis=1)
y = data["class"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

In [ ]:
y.head()

## 7. ✂️ Train-Test Split

Divide the data into training and testing subsets.

### Configuration
- **80%** → training data
- **20%** → testing data
- `random_state=42` → reproducible results
- `stratify=y` → preserves class proportions as much as possible

The test set remains unseen during model training.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 8. ⚖️ Handle Class Imbalance Using SMOTE

The disease classes are not equally represented.

**SMOTE (Synthetic Minority Over-sampling Technique)** creates synthetic training samples for minority classes.

### Important Rule
SMOTE is applied **only to the training data**. The original test data is kept untouched so that evaluation reflects the model's performance on unseen data.


In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print("Before SMOTE:", Counter(y_train))

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print("After SMOTE :", Counter(y_resampled))

In [ ]:
X_resampled.describe()

## 9. 📏 Feature Scaling

Standardize the feature values using `StandardScaler`.

### Why scaling is useful
Scaling is especially important for distance- and margin-based models such as:
- Logistic Regression
- SVM
- KNN

The scaler is fitted on the resampled training data and then applied to the test data.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_resampled)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_resampled.columns
)

X_train_scaled_df.describe()

## 10. 🧮 Logistic Regression

Train a Logistic Regression classifier as a baseline multi-class model.

**Purpose:** Establish a strong, interpretable baseline against which the other classifiers can be compared.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model_lr = LogisticRegression(max_iter=2000)
model_lr.fit(X_train_scaled, y_resampled)

y_pred_lr = model_lr.predict(X_test_scaled)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
print("Accuracy of Logistic Regression:", round(accuracy_lr * 100, 2), "%")

## 11. 🟣 Support Vector Machine (SVM)

Train an SVM classifier using an RBF kernel.

**Purpose:** Compare a margin-based nonlinear classifier with Logistic Regression and the other tree/distance-based approaches.


In [ ]:
from sklearn.svm import SVC

model_svc = SVC(kernel="rbf")
model_svc.fit(X_train_scaled, y_resampled)

y_pred_svc = model_svc.predict(X_test_scaled)

accuracy_svc = accuracy_score(y_test, y_pred_svc)
print("Accuracy of SVM:", round(accuracy_svc * 100, 2), "%")

## 12. 🌳 Decision Tree Classifier

Train a Decision Tree classifier.

**Strength:** Tree-based models can capture nonlinear relationships without requiring linear decision boundaries.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train_scaled, y_resampled)

y_pred_dt = model_dt.predict(X_test_scaled)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("Accuracy of Decision Tree:", round(accuracy_dt * 100, 2), "%")

## 13. 📍 K-Nearest Neighbors (KNN)

Train a KNN classifier.

**Key idea:** KNN predicts a class using the classes of nearby training observations, which makes feature scaling important.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier()
model_knn.fit(X_train_scaled, y_resampled)

y_pred_knn = model_knn.predict(X_test_scaled)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
print("Accuracy of KNN:", round(accuracy_knn * 100, 2), "%")

## 14. 🌲 Random Forest Classifier

Train a Random Forest classifier using multiple decision trees.

**Purpose:** Evaluate an ensemble tree-based approach and later inspect its feature importance values.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model_rf.fit(X_train_scaled, y_resampled)

y_pred_rf = model_rf.predict(X_test_scaled)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Accuracy of Random Forest:", round(accuracy_rf * 100, 2), "%")

## 15. 🏆 Compare All Models

Collect the accuracy of all trained classifiers in one DataFrame.

This provides a simple side-by-side comparison before the detailed evaluation of the Random Forest model.


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM",
        "Decision Tree",
        "KNN",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_lr,
        accuracy_svc,
        accuracy_dt,
        accuracy_knn,
        accuracy_rf
    ]
})

results["Accuracy (%)"] = (results["Accuracy"] * 100).round(2)
results.sort_values("Accuracy", ascending=False)

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x="Accuracy (%)", y="Model", data=results.sort_values("Accuracy (%)", ascending=False))
plt.title("Classification Model Accuracy Comparison")
plt.xlim(0, 100)
plt.show()

## 16. 🔬 Detailed Evaluation of Random Forest

Evaluate the Random Forest model using a classification report and confusion matrix.

### Metrics
- **Precision** — how many predicted samples for a class were actually that class.
- **Recall** — how many actual samples of a class were correctly identified.
- **F1-score** — harmonic mean of precision and recall.
- **Accuracy** — proportion of all predictions that are correct.

For multi-class data, the classification report provides class-wise results.


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score
)

print("Classification Report - Random Forest")
print(classification_report(y_test, y_pred_rf))

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

### 📐 Evaluation Metrics

- **Accuracy:** Percentage of all predictions that are correct.
- **Precision:** Of the samples predicted as a class, how many actually belong to that class.
- **Recall:** Of the samples that actually belong to a class, how many were correctly identified.
- **F1-score:** Harmonic mean of precision and recall.

For an imbalanced multi-class problem, it is useful to inspect more than accuracy alone.


In [ ]:
precision_rf = precision_score(y_test, y_pred_rf, average="weighted")
recall_rf = recall_score(y_test, y_pred_rf, average="weighted")
f1_rf = f1_score(y_test, y_pred_rf, average="weighted")

print("Random Forest Metrics")
print("Accuracy :", round(accuracy_rf * 100, 2), "%")
print("Precision:", round(precision_rf * 100, 2), "%")
print("Recall   :", round(recall_rf * 100, 2), "%")
print("F1 Score :", round(f1_rf * 100, 2), "%")

## 17. ⭐ Random Forest Feature Importance

Identify which input features contribute most strongly to the trained Random Forest's split decisions.

**Use:** This provides model-specific interpretability and helps highlight features that the trained model relied on most.


In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model_rf.feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10,7))
sns.barplot(
    x="Importance",
    y="Feature",
    data=feature_importance.head(15)
)
plt.title("Top 15 Important Features - Random Forest")
plt.show()

## 18. 💾 Save the Trained Model and Scaler

Save the trained Random Forest model and fitted scaler so they can be reused outside the notebook.

These files can later support:
- A Flask/Streamlit application
- A prediction API
- A simple desktop/web interface
- Reproducible inference


In [ ]:
import pickle

pickle.dump(model_rf, open("dermatology_random_forest.pkl", "wb"))
pickle.dump(scaler, open("dermatology_scaler.pkl", "wb"))

print("Model saved as dermatology_random_forest.pkl")
print("Scaler saved as dermatology_scaler.pkl")

## 19. 🔮 Test a New Prediction

Load the saved model and scaler and use them to generate a prediction for a new input.

### Prediction Flow
**User/feature values → preprocessing → scaling → trained model → predicted disease class**

> This prediction interface is for project demonstration and educational purposes only.


In [ ]:
import numpy as np
import pickle

# Load saved model and scaler
with open("dermatology_random_forest.pkl", "rb") as f:
    model = pickle.load(f)

with open("dermatology_scaler.pkl", "rb") as f:
    scaler_loaded = pickle.load(f)

feature_names = list(X.columns)

print("Enter values for the following features:")

user_values = []

for feature in feature_names:
    value = float(input(f"Enter {feature}: "))
    user_values.append(value)

input_data = np.array(user_values).reshape(1, -1)

scaled_input = scaler_loaded.transform(input_data)

prediction = model.predict(scaled_input)

print("\nPredicted Disease Class:", prediction[0])

## 20. ✅ Project Conclusion

The Dermatology dataset was used to build a **multi-class classification workflow** using several machine learning algorithms.

### Workflow Completed
1. Loaded and inspected the dataset.
2. Checked shape, columns, missing values, and duplicates.
3. Converted the `age` feature to numeric form and handled missing values.
4. Performed exploratory data analysis.
5. Separated features and target.
6. Created training and testing sets.
7. Applied SMOTE only to the training data.
8. Standardized the features.
9. Trained five classification algorithms.
10. Compared model accuracy.
11. Evaluated the Random Forest model in detail.
12. Examined feature importance.
13. Saved the trained model and scaler.
14. Tested a new prediction.

### 🚀 Optional Extensions Added Below
The following cells extend the same project without changing the original workflow:
- Cross-validation comparison
- Precision/recall/F1 model comparison
- Confusion matrices for all models
- Random Forest probability-based confidence display
- Prediction helper function with feature validation
- Model metadata export
- Reproducibility checks
- Deployment-ready inference example

### ⚠️ Academic Note
The trained model is a machine-learning demonstration and must not be interpreted as a clinical diagnosis system.


## 21. 🎓 Viva Questions — Short Answers

Use these questions for quick project revision and viva preparation.


# 🚀 22. Project Extensions — Advanced Analysis

The following cells are **additional enhancements**. They do not replace or reorder the original project steps.

## Extension A — Cross-Validation
Use stratified cross-validation on the original training data to get a more stable estimate of model performance.

> This extension uses the already prepared `X_train`, `y_train`, and the existing classifier definitions. It is an additional validation view, not a replacement for the original test-set evaluation.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "SVM": SVC(kernel="rbf"),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

cv_results = []

for name, estimator in cv_models.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    cv_results.append({
        "Model": name,
        "CV Mean Accuracy": scores.mean(),
        "CV Std": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    "CV Mean Accuracy", ascending=False
)

display(cv_results_df.style.format({
    "CV Mean Accuracy": "{:.4f}",
    "CV Std": "{:.4f}"
}))


## Extension B — Compare Precision, Recall and F1-Score

Accuracy alone may not fully describe a multi-class classifier. This extension compares weighted precision, recall and F1-score for every trained model.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

model_predictions = {
    "Logistic Regression": y_pred_lr,
    "SVM": y_pred_svc,
    "Decision Tree": y_pred_dt,
    "KNN": y_pred_knn,
    "Random Forest": y_pred_rf
}

metric_rows = []

for name, predictions in model_predictions.items():
    metric_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, predictions, average="weighted", zero_division=0),
        "F1-Score": f1_score(y_test, predictions, average="weighted", zero_division=0)
    })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-Score": "{:.4f}"
}))


## Extension C — Confusion Matrices for All Models

Visualize the prediction pattern of every classifier. The diagonal represents correctly classified observations, while off-diagonal cells show class confusions.


In [ ]:
for name, predictions in model_predictions.items():
    cm_model = confusion_matrix(y_test, predictions)

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm_model, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.tight_layout()
    plt.show()


## Extension D — Random Forest Prediction Probabilities

Random Forest can provide class probabilities using `predict_proba()`.

The highest probability is used only as a model output indicator. It should **not** be interpreted as a medical certainty or clinical confidence.


In [ ]:
# Probability distribution for the first five test samples
rf_probabilities = model_rf.predict_proba(X_test_scaled)
rf_probability_df = pd.DataFrame(
    rf_probabilities,
    columns=[f"Class_{c}" for c in model_rf.classes_]
)

display(rf_probability_df.head())


## Extension E — Reusable Prediction Function

Create a reusable function that checks feature names, applies the saved scaler, and returns the predicted class.

This makes the notebook easier to connect to a future Flask or Streamlit application.


In [ ]:
def predict_dermatology_case(feature_values, model=model_rf, fitted_scaler=scaler, feature_columns=None):
    """Predict a dermatology class from one feature record.

    Parameters
    ----------
    feature_values : dict
        Dictionary containing all required feature names and values.
    model : trained classifier
        Trained classification model.
    fitted_scaler : fitted scaler
        Scaler fitted during the training workflow.
    feature_columns : list
        Expected feature order.

    Returns
    -------
    dict
        Predicted class and, when available, class probabilities.
    """
    if feature_columns is None:
        feature_columns = list(X.columns)

    missing = [col for col in feature_columns if col not in feature_values]
    extra = [col for col in feature_values if col not in feature_columns]

    if missing:
        raise ValueError(f"Missing features: {missing}")
    if extra:
        raise ValueError(f"Unexpected features: {extra}")

    input_df = pd.DataFrame(
        [[feature_values[col] for col in feature_columns]],
        columns=feature_columns
    )

    input_scaled = fitted_scaler.transform(input_df)
    predicted_class = model.predict(input_scaled)[0]

    result = {"predicted_class": predicted_class}

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(input_scaled)[0]
        result["class_probabilities"] = dict(
            zip(model.classes_, probabilities)
        )

    return result

print("Prediction function created successfully.")


## Extension F — Save Model Metadata

Store the feature order and basic model information alongside the model files.

Keeping the feature order is important when the model is later used by an external application.


In [ ]:
import json

model_metadata = {
    "model_name": "Random Forest",
    "target_column": "class",
    "feature_count": len(X.columns),
    "feature_names": list(X.columns),
    "random_state": 42,
    "model_file": "dermatology_random_forest.pkl",
    "scaler_file": "dermatology_scaler.pkl"
}

with open("dermatology_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=4)

print("Saved: dermatology_model_metadata.json")
print("Number of features:", model_metadata["feature_count"])


## Extension G — Project Readiness Checklist

Use this final checklist before presenting or deploying the project.

- [x] Dataset loaded
- [x] Missing values handled
- [x] Duplicate records checked
- [x] Class distribution inspected
- [x] EDA completed
- [x] Train/test split completed
- [x] SMOTE applied only to training data
- [x] Feature scaling completed
- [x] Multiple classifiers trained
- [x] Models compared
- [x] Random Forest evaluated
- [x] Feature importance inspected
- [x] Model and scaler saved
- [x] Reusable prediction function added
- [x] Cross-validation added
- [x] Additional evaluation metrics added
- [x] Model metadata saved

### 🌐 Next Practical Step
The saved model can now be connected to a **Flask or Streamlit user interface** that accepts feature values and displays the predicted class.

**Reminder:** This project is for educational/model-development purposes and should not be presented as a clinically validated diagnostic tool.
